# A3.1 · Default-deny on the tool call

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A2.8 · An audit trail the workload cannot forge](https://spbreed.github.io/cyber-commons/lessons/A2.8.html)**.

| | |
|---|---|
| Tools used | OPA, kmcp |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Identity has already failed. Something untrusted is in the context and the agent has decided to call a tool. The tool call is the last place a decision can still be made on facts rather than on intent, and it is where default-deny belongs.

> **At CyberTravels.** The last place a decision about that refund rests on facts rather than on intent. Identity has already failed, an injected instruction is in the context, and the tool call is where CyberTravels can still say no. R1, R3.

## 2 · The framework

```
   untrusted text in context ---> agent decides to call a tool
                                             |
                                    +--------v---------+
                                    |  policy decision |
                                    |  DEFAULT: DENY   |
                                    +--------+---------+
                                             |
                        allow only on facts: identity, scope,
                        resource, provenance of the motivating span

   the last point where a decision rests on facts rather than on intent
```

**Mitigates: T2 Tool Misuse · T3 Privilege Compromise · T6 Intent Breaking.**

The tool call is the moment text becomes consequence. It is also the last point
where a decision can be made on **facts** — this identity, this tool, these
arguments, this resource — rather than on intent, which nobody can read.

Default-deny means the absence of a rule is a refusal. That sounds like a
detail and it is the entire control, because it changes what a mistake costs.
Under allow-by-default, a permission somebody forgot to restrict is available to
an attacker. Under deny-by-default, a permission somebody forgot to grant is a
broken feature — which someone reports on Monday morning, loudly, and which
harms nobody.

The policy takes four inputs and all four matter:

- **identity** — the workload, from A2.1
- **tool** — which capability
- **arguments** — the actual values, not the schema
- **resource** — which specific thing

Dropping the fourth is the most common weakening. `run_query` permitted for the
reports agent is not the same as `run_query` permitted *on the reports table*,
and A1.5 was the difference between those two sentences.

This does not stop the agent being persuaded. It stops persuasion mattering,
which is a better place to stand.

> **What this control closes.**
>
> Stands on the edge every topology shares: `agent_runtime -> tools`. Persuasion still happens; it just stops reaching anything.

## 3 · The control

In [ ]:
POLICY = [
 # (identity,        tool,        resource-prefix,   allowed-args)
 ("reports-agent", "run_query",  "table:reports",   {"SELECT"}),
 ("reports-agent", "send_email", "domain:corp.example", {"*"}),
 ("billing-agent", "run_query",  "table:invoices",  {"SELECT", "UPDATE"}),
]

def decide(identity, tool, resource, verb, default_deny=True):
    """Four inputs. No matching rule means refuse."""
    for ident, t, res_prefix, verbs in POLICY:
        if ident == identity and t == tool and resource.startswith(res_prefix):
            if "*" in verbs or verb in verbs:
                return True, f"rule {ident}/{t}/{res_prefix}"
            return False, f"verb {verb} not permitted on {res_prefix}"
    return (False, "no rule matches - default deny") if default_deny else (True, "allowed by default")

CALLS = [
 ("reports-agent", "run_query",  "table:reports",  "SELECT"),   # intended
 ("reports-agent", "run_query",  "table:secrets",  "SELECT"),   # A1.5, resource
 ("reports-agent", "run_query",  "table:reports",  "DELETE"),   # A1.5, verb
 ("reports-agent", "send_email", "domain:evil.example", "*"),   # A1.3, exfil
 ("reports-agent", "drop_table", "table:reports",  "*"),        # tool never granted
]

for mode in (False, True):
    label = "DEFAULT-DENY" if mode else "allow-by-default"
    allowed = 0
    print(f"{label}:")
    for identity, tool, resource, verb in CALLS:
        ok, why = decide(identity, tool, resource, verb, default_deny=mode)
        allowed += ok
        print(f"   {tool:11s}{resource:22s}{verb:7s}{'ALLOW' if ok else 'deny ':6s}{why}")
    print(f"   -> {allowed}/{len(CALLS)} permitted\n")

print("The only call that should succeed is the first. Under allow-by-default")
print("four do, and each one is a real risk from Chapter 1 walking through.")
print()
print("Note the third row: same identity, same tool, same resource, refused on")
print("the verb. Authorization attached to the tool instead of the call cannot")
print("express that distinction at all.")
assert sum(decide(*c, default_deny=True)[0] for c in CALLS) == 1

## What you just proved

Five tool calls are evaluated twice. Under allow-by-default four succeed, each one a Chapter 1 risk walking through. Under default-deny only the intended call survives — including a refusal on the verb for an otherwise-permitted identity, tool and resource.

## Your turn

Take one tool policy you have and check whether it names the resource. If it grants `run_query` rather than `run_query on these tables`, it cannot express the difference that A1.5 turned on.

---

**Next → [A3.2 · Sandboxed execution](https://spbreed.github.io/cyber-commons/lessons/A3.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*